# objetivo específico 1 “verificar que usar el índice del mes anterior no cambia los resultados”.

In [1]:
%run 1_home.ipynb

In [2]:
# IPC nacional, INDEC via datos.gob.ar (serie 148.3_INIVELNAL_DICI_M_26)
IPC = {
    201901: 189.6101, 201902: 196.7501, 201903: 205.9571, 201904: 213.0517,
    201905: 219.5691, 201906: 225.5370, 201907: 230.4940, 201908: 239.6077,
    201909: 253.7102, 201910: 262.0661, 201911: 273.2158, 201912: 283.4442,
}
IPC_DIC2018 = 184.2552      # el mes previo al inicio de la serie
MES_REFERENCIA = 201901

# Cada mes deflactado con el indice del mes ANTERIOR, que es el unico
# que estaria publicado al momento de scorear.
meses = sorted(IPC)
IPC_REZAGADO = {m: (IPC_DIC2018 if i == 0 else IPC[meses[i - 1]])
                for i, m in enumerate(meses)}

def columnas_monetarias(df):
    return [c for c in df.columns
            if c.startswith("m") or c.startswith("Master_m") or c.startswith("Visa_m")]

def deflactar(df, indice, mes_referencia=MES_REFERENCIA, columnas=None):
    """monto / indice del mes * indice del mes de referencia"""
    out = df.copy()
    columnas = columnas or columnas_monetarias(df)
    factor = df.foto_mes.map(indice).astype("float32")
    out[columnas] = out[columnas].astype("float32").div(factor, axis=0) * indice[mes_referencia]
    return out

for m in meses:
    print(m, round(IPC[m], 2), "->", round(IPC_REZAGADO[m], 2))

201901 189.61 -> 184.26
201902 196.75 -> 189.61
201903 205.96 -> 196.75
201904 213.05 -> 205.96
201905 219.57 -> 213.05
201906 225.54 -> 219.57
201907 230.49 -> 225.54
201908 239.61 -> 230.49
201909 253.71 -> 239.61
201910 262.07 -> 253.71
201911 273.22 -> 262.07
201912 283.44 -> 273.22


In [3]:
df = cargar_datos()
df_rez = deflactar(df, IPC_REZAGADO)
datos, features = agregar_lags_deltas(df_rez)
print(datos.shape, len(features))

(1563565, 763) 760


In [4]:
import os, shutil, optuna

ESTUDIO_LOCAL  = "/home/ds/datasets/optuna_ipc.db"
ESTUDIO_BUCKET = "/home/ds/buckets/b1/optuna_ipc.db"

if not os.path.exists(ESTUDIO_LOCAL):
    shutil.copy(ESTUDIO_BUCKET, ESTUDIO_LOCAL)

# Se reusan tal cual los hiperparametros del brazo 3. Es un chequeo de
# sensibilidad: si se vuelve a optimizar, no se sabe si la diferencia
# viene del indice o de la busqueda.
estudio_ipc = optuna.load_study(study_name="deflactado_ipc",
                                storage=f"sqlite:///{ESTUDIO_LOCAL}")
print(estudio_ipc.best_params)
print("rondas:", estudio_ipc.best_trial.user_attrs["mejor_iteracion"])

{'num_leaves': 255, 'learning_rate': 0.011433353105812017, 'min_data_in_leaf': 2156, 'feature_fraction': 0.8809664444631677, 'bagging_fraction': 0.8869992712494833}
rondas: 453


In [ ]:
res_rez, curvas_rez, probas_rez = correr_semillas(datos, features, estudio_ipc)
res_rez.to_csv("/home/ds/buckets/b1/rezagado_5semillas.csv", index=False)
res_rez

In [ ]:
res_ipc = pd.read_csv("/home/ds/buckets/b1/ipc_5semillas.csv")
dif = comparar(res_ipc, res_rez, "indice del mes", "indice del mes anterior")

print("\nbanda de ruido del nominal: $4,650,000")
print("entra en el ruido:", abs(dif.mean()) < 4_650_000)